# Notebook 04: SAR Flood Detection

Explores Sentinel-1 SAR backscatter analysis for detecting surface water inundation.

Key concepts:
- VV polarisation sensitivity to surface roughness
- Open water → very smooth surface → specular reflection away from sensor → low VV dB
- Delta VV between pre-flood and flood period flags inundated pixels

In [ ]:
import sys; sys.path.insert(0, '..')
from dotenv import load_dotenv; load_dotenv('../.env')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from apps.api.services.imagery import SentinelHubFetcher
from apps.api.services.change.flood import FloodDetector

In [ ]:
# Sundarbans Delta — flood-prone region
BBOX = [88.3, 21.5, 89.0, 22.1]

fetcher = SentinelHubFetcher()

# Pre-monsoon (dry season) vs monsoon
sar_before = fetcher.get_sentinel1_composite(BBOX, '2023-03-01', '2023-03-31')
sar_after  = fetcher.get_sentinel1_composite(BBOX, '2023-08-01', '2023-08-31')

# S2 for urban mask
s2_after = fetcher.get_sentinel2_composite(BBOX, '2023-08-01', '2023-08-31')

print(f'SAR before shape: {sar_before.shape}  (VV=ch0, VH=ch1)')
print(f'SAR after  shape: {sar_after.shape}')

In [ ]:
# VV backscatter statistics
vv_before = sar_before[:, :, 0]
vv_after  = sar_after[:, :, 0]
vv_delta  = vv_before - vv_after  # positive = signal decrease = potential flood

print(f'VV before: mean={vv_before.mean():.2f} dB  std={vv_before.std():.2f}')
print(f'VV after:  mean={vv_after.mean():.2f}  dB  std={vv_after.std():.2f}')
print(f'VV delta:  mean={vv_delta.mean():.2f}  dB  std={vv_delta.std():.2f}')

In [ ]:
# Run flood detector
detector = FloodDetector(vv_decrease_threshold=4.0)
mask, confidence = detector.detect(sar_before, sar_after, after_s2=s2_after)

print(f'Flood mask: {mask.sum():,} pixels ({100*mask.mean():.2f}% of image)')
print(f'Confidence: {confidence:.3f}')

In [ ]:
# Visualise SAR before/after and flood mask
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

vmin, vmax = -20, 0  # dB range for VV

im0 = axes[0].imshow(vv_before, cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('VV Backscatter — Before (dB)', fontsize=10)
axes[0].axis('off')
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(vv_after, cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('VV Backscatter — After (dB)', fontsize=10)
axes[1].axis('off')
fig.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(vv_delta, cmap='RdBu_r', vmin=-10, vmax=10)
axes[2].set_title('VV Delta (Before - After)', fontsize=10)
axes[2].axis('off')
fig.colorbar(im2, ax=axes[2], fraction=0.046)

axes[3].imshow(vv_after, cmap='gray', vmin=vmin, vmax=vmax)
axes[3].imshow(mask, cmap='Blues', alpha=0.6, vmin=0, vmax=1)
axes[3].set_title(f'Flood Mask (conf={confidence:.2f})', fontsize=10)
axes[3].axis('off')

plt.suptitle('SAR Flood Detection — Sundarbans', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('04_sar_flood_detection.png', dpi=150)
plt.show()

In [ ]:
# Sensitivity analysis: vary VV threshold
thresholds = [2.0, 3.0, 4.0, 5.0, 6.0, 8.0]
results = []

for thr in thresholds:
    det = FloodDetector(vv_decrease_threshold=thr)
    m, c = det.detect(sar_before, sar_after)
    results.append({'threshold_dB': thr, 'pct_flagged': 100*m.mean(), 'confidence': c})

import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(df['threshold_dB'], df['pct_flagged'], 'b-o', label='% pixels flagged')
ax2.plot(df['threshold_dB'], df['confidence'], 'r--s', label='Confidence')
ax1.set_xlabel('VV Decrease Threshold (dB)')
ax1.set_ylabel('% Pixels Flagged', color='b')
ax2.set_ylabel('Confidence', color='r')
ax1.legend(loc='upper right')
ax2.legend(loc='center right')
plt.title('Flood Detector Sensitivity to VV Threshold')
plt.tight_layout()
plt.show()